# Tugas 6 - Sistem pencarian dokumen menggunakan Singular Value Decomposition (SVD)

**Nama : Achmad Baharuddin Akbar**

**NIM  : 210411100001**


- sistem pencarian dokumen
- reduksi dimensi dengan SVD
- data baru juga di reduksi
- mencari kemiripan ecludian distance

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [3]:
import pandas as pd

data_path = ("/content/drive/My Drive/PPW-A/report/tugas-ppw/hasil_preprocesing.csv")
news_data = pd.read_csv(data_path)

news_data

,judul,tanggal,isi,kategori,cleansing,case_folding,tokenize,Filtering/stopword removal
0,Gejala Sifilis pada Wanita Berdasarkan Tahapan...,"Rabu, 16 Okt 2024 21:00 WIB",Jakarta - Sifilis atau penyakit raja singa ter...,Kesehatan,Jakarta Sifilis atau penyakit raja singa term...,jakarta sifilis atau penyakit raja singa term...,"['jakarta', 'sifilis', 'atau', 'penyakit', 'ra...",jakarta sifilis penyakit raja singa infeksi me...
1,Puncak Nafsu Pria Ada di Umur Berapa? Studi Bi...,"Rabu, 16 Okt 2024 20:02 WIB",Jakarta - Sebagian orang pasti sudah mengenal ...,Kesehatan,Jakarta Sebagian orang pasti sudah mengenal i...,jakarta sebagian orang pasti sudah mengenal i...,"['jakarta', 'sebagian', 'orang', 'pasti', 'sud...",jakarta orang mengenal istilah puncak seksual ...
2,"Gejala Kanker Mulut yang Kerap Tak Disadari, T...","Rabu, 16 Okt 2024 18:02 WIB",Jakarta - Setiap orang mungkin pernah mengalam...,Kesehatan,Jakarta Setiap orang mungkin pernah mengalami...,jakarta setiap orang mungkin pernah mengalami...,"['jakarta', 'setiap', 'orang', 'mungkin', 'per...",jakarta orang mengalami sariawan luka mulut me...
3,Kapan Waktu yang Tepat untuk Minum Air Rebusan...,"Rabu, 16 Okt 2024 17:31 WIB",Jakarta - Air rebusan serai dikenal sebagai sa...,Kesehatan,Jakarta Air rebusan serai dikenal sebagai sal...,jakarta air rebusan serai dikenal sebagai sal...,"['jakarta', 'air', 'rebusan', 'serai', 'dikena...",jakarta air rebusan serai dikenal salah ramuan...
4,Viral Hanni NewJeans Bicara soal Bullying di T...,"Rabu, 16 Okt 2024 16:34 WIB","Jakarta - Viral momen member NewJeans, Hanni, ...",Kesehatan,Jakarta Viral momen member NewJeans Hanni ber...,jakarta viral momen member newjeans hanni ber...,"['jakarta', 'viral', 'momen', 'member', 'newje...",jakarta viral momen member newjeans hanni berb...
...,...,...,...,...,...,...,...,...
95,Lamaknyo! Soto Padang Isian Daging Goreng Ada ...,"Selasa, 15 Okt 2024 12:00 WIB",Jakarta - Tak kalah populer dengan jenis soto ...,Kuliner,Jakarta Tak kalah populer dengan jenis soto l...,jakarta tak kalah populer dengan jenis soto l...,"['jakarta', 'tak', 'kalah', 'populer', 'dengan...",jakarta kalah populer jenis soto soto padang k...
96,Duh! Kedai Lumpia Ini Ditutup 14 Hari Usai Tem...,"Selasa, 15 Okt 2024 11:30 WIB",Jakarta - Kedai yang terkenal dengan hidangan ...,Kuliner,Jakarta Kedai yang terkenal dengan hidangan l...,jakarta kedai yang terkenal dengan hidangan l...,"['jakarta', 'kedai', 'yang', 'terkenal', 'deng...",jakarta kedai terkenal hidangan lumpia terpaks...
97,"Resep Cempedak Goreng Krispi, Manis Renyah Unt...","Selasa, 15 Okt 2024 11:00 WIB",Tentang Bahan Langkah Jakarta - Cempedak yang ...,Kuliner,Tentang Bahan Langkah Jakarta Cempedak yang m...,tentang bahan langkah jakarta cempedak yang m...,"['tentang', 'bahan', 'langkah', 'jakarta', 'ce...",bahan langkah jakarta cempedak manis legit har...
98,10 Tampilan Makanan yang Disebut Buruk Rupa Ta...,"Selasa, 15 Okt 2024 10:30 WIB",Jakarta - Meski tampilan beberapa makanan ini ...,Kuliner,Jakarta Meski tampilan beberapa makanan ini d...,jakarta meski tampilan beberapa makanan ini d...,"['jakarta', 'meski', 'tampilan', 'beberapa', '...",jakarta tampilan makanan dianggap buruk rupa a...


In [4]:
# Ambil kolom dokumen berita
documents = news_data['Filtering/stopword removal'].tolist()

**TF-IDF (Term Frequency-Inverse Document Frequency)**

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import euclidean_distances
import pandas as pd
import numpy as np
import joblib

In [6]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

# Mendapatkan fitur (kata-kata) dari TF-IDF
tfidf_features = tfidf_vectorizer.get_feature_names_out()

# Mengonversi matriks TF-IDF menjadi DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_features)

# Tampilkan hasil TF-IDF
print("\nHasil TF-IDF dalam bentuk DataFrame:")
tfidf_df


Hasil TF-IDF dalam bentuk DataFrame:


,abai,abang,abc,abdalla,abdullah,abis,abnormal,acar,acara,aceh,...,you,youtuber,yu,yudhistira,yun,yuni,zaitun,zaman,zat,zonk
0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.026415,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
96,0.0,0.0,0.0,0.0,0.044578,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
97,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
98,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


**Singular Value Decomposition (SVD)**

In [8]:
# Menggunakan SVD untuk reduksi dimensi
n_components = 99
svd = TruncatedSVD(n_components=n_components)
svd_matrix = svd.fit_transform(tfidf_matrix)

# Mengubah hasil SVD menjadi DataFrame
svd_df = pd.DataFrame(svd_matrix, columns=[f'feature {i+1}' for i in range(n_components)])
svd_df['kategori'] = news_data['kategori'].values

# Menampilkan hasil
print("Hasil reduksi dimensi dengan SVD:")
svd_df

Hasil reduksi dimensi dengan SVD:


,feature 1,feature 2,feature 3,feature 4,feature 5,feature 6,feature 7,feature 8,feature 9,feature 10,...,feature 91,feature 92,feature 93,feature 94,feature 95,feature 96,feature 97,feature 98,feature 99,kategori
0,0.106579,-0.051003,-0.034821,-0.094111,0.014064,0.088596,-0.062249,-0.082666,0.050542,-0.125903,...,0.010893,-0.007947,0.007142,-0.000464,-0.000241,0.001081,0.001296,-0.000029,-0.000431,Kesehatan
1,0.092157,-0.069143,0.007447,-0.076963,-0.002376,0.067781,-0.041185,-0.069199,0.026778,-0.065221,...,-0.006301,0.001103,-0.004274,-0.011882,-0.001521,0.004413,0.000671,-0.000179,0.001475,Kesehatan
2,0.122358,-0.093063,-0.035451,-0.122221,-0.015544,0.159948,-0.159300,0.024506,-0.015228,-0.057546,...,-0.009153,0.000059,-0.001523,-0.002292,0.006131,0.001729,-0.002822,0.001783,-0.001304,Kesehatan
3,0.145855,-0.161630,-0.049319,-0.102135,-0.010724,-0.089639,-0.044322,0.026075,0.141075,-0.113237,...,-0.007732,-0.003578,0.003297,-0.007511,0.001122,-0.001559,-0.002996,0.001071,0.000862,Kesehatan
4,0.061430,-0.000351,0.014411,-0.002673,0.000036,0.056426,-0.042888,-0.007846,0.013437,0.031302,...,-0.004928,-0.001055,0.007543,-0.000842,-0.003827,0.000580,0.000052,0.002567,-0.006066,Kesehatan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.113692,-0.146180,0.012188,0.150142,-0.008660,-0.053105,-0.030566,-0.003122,-0.076397,-0.040679,...,-0.002036,0.012274,0.001764,-0.003378,-0.000834,-0.001078,-0.002710,0.006907,-0.000088,Kuliner
96,0.098431,-0.062604,0.010040,0.076850,0.007179,0.061454,0.030421,0.008962,0.104330,0.047030,...,0.004413,0.004392,0.002256,-0.003980,0.000837,-0.001564,-0.006473,-0.006269,-0.001007,Kuliner
97,0.115704,-0.178179,0.010109,0.061058,-0.023253,-0.288983,-0.148836,0.114466,0.148672,0.218154,...,0.004376,0.018191,0.005424,0.003845,-0.003030,0.001985,0.002288,0.001671,0.000961,Kuliner
98,0.102689,-0.101178,-0.016928,0.062199,0.007463,0.043892,-0.010565,-0.132709,0.047946,-0.016223,...,-0.007745,-0.011627,0.005266,-0.000646,0.000217,-0.001156,-0.003092,-0.006019,-0.003432,Kuliner


**Implementasi**

In [9]:
def search_similar_documents(query_text, top_k=5):

    # Transformasi query text
    query_tfidf = tfidf_vectorizer.transform([query_text])
    query_svd = svd.transform(query_tfidf)

    # Hitung jarak Euclidean antara query dan dokumen
    distances = euclidean_distances(query_svd, svd_matrix)

    # Ambil indeks dokumen terdekat
    closest_indices = np.argsort(distances[0])[:top_k]
    similarities = distances[0][closest_indices]

    return closest_indices, similarities

In [10]:
# Fungsi untuk melakukan preprocessing
def preprocess_text(text):

    # Cleaning: Menghapus angka dan tanda baca
    text = re.sub(r'\d+', '', text)  # Hapus angka
    text = text.translate(str.maketrans('', '', string.punctuation))  # Hapus tanda baca

    # Casefolding: Mengubah ke huruf kecil
    text = text.lower()

    # Tokenizing
    words = word_tokenize(text)

    # Stopword removal
    stop_words = set(stopwords.words("indonesian"))  # Ganti dengan 'english' jika teks berbahasa Inggris
    words = [word for word in words if word not in stop_words]

    # Gabungkan kembali menjadi string
    return ' '.join(words)

# Input teks dan preprocessing
query_text = input("Masukkan teks: ")
processed_query_text = preprocess_text(query_text)

# Lakukan pencarian dengan teks yang telah dipreproses
top_k = 5  # Jumlah dokumen yang diinginkan
indices, distances = search_similar_documents(processed_query_text, top_k)

# Menampilkan hasil pencarian
print("\nHasil pencarian untuk query:", processed_query_text)
for idx, distance in zip(indices, distances):
    print(f"Dokumen ke-{idx + 1}: {documents[idx]}")
    print(f"Jarak Euclidean: {distance:.4f}\n")

Masukkan teks: a

Hasil pencarian untuk query: a
Dokumen ke-32: jakarta penelitian golongan darah berisiko terkena stroke usia muda stroke pembuluh darah tersumbat pecah kondisi disebabkan faktor kesehatan gaya hidup buruk dasarnya kondisi dialami usia anak muda rentang usia terkena stroke penelitian membuktikan anak muda terkena stroke berusia memiliki golongan darah a dikutip science alert penelitian hubungan gen golongan darah a risiko stroke penelitian tim ilmuwan fakultas kedokteran maryland university mengamati hubungan karakteristik genetik golongan darah stroke peneliti mengambil data studi genetik stroke iskemik orang dewasa berusia mengidap stroke orang pengidap stroke diteliti pencarian genom luas mengungkap area kromosom terkait risiko stroke area bertepatan gen golongan darah hasil analisis orang golongan darah a berpotensi persen terkena stroke usia golongan darah untungnya memiliki persen risiko terkena stroke tambahan penelitian menemukan orang golongan darah b berpoten

****

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Menggunakan TF-IDF untuk transformasi teks
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(documents)

# Memisahkan dataset menjadi data training dan testing
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, news_data['kategori'], test_size=0.2, random_state=42)

# Melakukan reduksi dimensi pada data TF-IDF menggunakan SVD
n_components = 80  # Mengatur jumlah komponen sesuai keinginan
svd = TruncatedSVD(n_components=n_components)
X_train_svd = svd.fit_transform(X_train)
X_test_svd = svd.transform(X_test)

In [40]:
# Mengubah hasil SVD menjadi DataFrame
svd = pd.DataFrame(X_train_svd, columns=[f'feature {i+1}' for i in range(n_components)])
svd

,feature 1,feature 2,feature 3,feature 4,feature 5,feature 6,feature 7,feature 8,feature 9,feature 10,...,feature 71,feature 72,feature 73,feature 74,feature 75,feature 76,feature 77,feature 78,feature 79,feature 80
0,0.149138,0.088192,-0.025737,-0.012817,0.028467,0.032598,0.094246,-0.128312,0.319420,-0.221137,...,0.006136,-0.001084,-0.041222,0.014467,0.016481,-0.016953,-0.008583,0.002420,0.000329,0.005554
1,0.104755,0.081044,0.011026,0.047268,-0.019506,0.027006,-0.035625,-0.081078,0.057197,-0.013999,...,0.002311,0.005698,0.039626,-0.011277,0.003770,0.003892,0.006602,0.001815,-0.001229,0.001761
2,0.122897,0.005459,0.129695,0.273201,-0.058263,0.125100,-0.006946,0.170399,-0.048914,-0.000011,...,0.031337,0.000836,-0.007190,0.011786,0.007391,-0.005116,-0.000370,0.000069,0.003178,0.000458
3,0.170052,0.024351,0.160846,0.327857,-0.096308,0.148069,-0.007637,0.177077,-0.011303,0.005027,...,-0.036495,0.036358,0.016827,-0.006368,-0.003179,-0.002580,-0.003020,-0.000416,-0.004958,0.000997
4,0.230973,0.131773,0.000427,0.042288,-0.114952,-0.068184,-0.115280,-0.195818,0.071897,0.045816,...,-0.072964,-0.021780,0.016458,0.007892,0.010239,0.000384,-0.012573,0.003441,-0.000198,0.001742
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0.112817,0.059486,-0.003667,0.003123,0.007889,0.020963,-0.015618,-0.101718,0.141282,-0.022103,...,-0.008354,-0.010461,0.019731,-0.008379,0.010436,-0.003760,-0.005495,0.005935,0.000880,0.002314
76,0.223728,0.279698,-0.113680,-0.249515,0.385201,0.706885,-0.156486,-0.033333,-0.126538,0.077864,...,-0.019261,-0.018870,-0.025896,-0.000626,0.000757,0.003125,-0.001942,0.003099,-0.012953,0.298372
77,0.225430,-0.146204,-0.038527,0.117017,0.054944,0.068957,0.550983,-0.308842,-0.410830,-0.293234,...,0.024875,0.017859,-0.017694,-0.021533,-0.411688,0.205095,-0.040040,0.002605,-0.001654,-0.000074
78,0.132725,0.076059,0.022150,-0.009855,-0.005591,-0.020663,0.121497,-0.090870,0.273695,-0.105995,...,-0.014825,0.002326,0.025843,0.001693,-0.010011,0.008448,-0.018275,-0.013871,-0.000305,-0.001389


In [39]:
# Membuat dan melatih model Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_svd, y_train)

# Memprediksi hasil pada data testing
y_pred = model.predict(X_test_svd)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

# Menampilkan confusion matrix dan laporan klasifikasi
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 80.00%
Confusion Matrix:
[[9 3]
 [1 7]]

Classification Report:
              precision    recall  f1-score   support

   Kesehatan       0.90      0.75      0.82        12
     Kuliner       0.70      0.88      0.78         8

    accuracy                           0.80        20
   macro avg       0.80      0.81      0.80        20
weighted avg       0.82      0.80      0.80        20

